In [0]:
%run "../config/schema_configs"

In [0]:
from pyspark.sql.functions import col, from_json, current_timestamp, row_number
from pyspark.sql.window import Window

print(f"Inicializando o Pipeline de Tratamento e Qualidade (Camada Silver)...")

cfg_click = SILVER_CONFIG['clickstream']
cfg_sales = SILVER_CONFIG['sales']

# =====
# --- PIPELINE clickstream
# =====

# leitura incremental da bronze
df_bronze_click = spark.readStream.table(cfg_click["source_table"])

# extrai os campos do JSON e cria colunas tipadas
df_silver_click_parsed = df_bronze_click.select(
    col("event_id").cast("string"),
    col("user_id").cast("string"),
    col("timestamp").cast("timestamp").alias("event_timestamp"),
    col("action").cast("string"),
    # Tratando device_info 
    col("device_info.os").cast("string").alias("device_os"),
    col("device_info.browser").cast("string").alias("device_browser"),
    # Tratando o payload interno
    col("payload.product_id").cast("string").alias("product_id"),
    col("payload.session_id").cast("string").alias("session_id"),
    # Capturando a evolução de schema (coluna nova que o Auto Loader mapeou no lote 3)
    col("payload.discount_applied").cast("boolean").alias("discount_applied"),
    col("ingestion_timestamp").alias("bronze_ingestion_timestamp")
)

# Técnica de Deduplicação usando micro-batch interno (foreachBatch)
# O streaming puro tem limitações com dropDuplicates() global se não houver watermark estrito.
# Usar foreachBatch garante controle absoluto de escrita idempotente linha por linha.
def upsert_clickstream_silver(df_micro_batch, batch_id):
    if df_micro_batch.count() == 0:
        return

    # Janela para identificar e remover duplicados dentro do próprio lote que está chegando
    window_spec = Window.partitionBy("event_id").orderBy(col("bronze_ingestion_timestamp").desc())
    
    df_deduped = df_micro_batch \
        .withColumn("row_num", row_number().over(window_spec)) \
        .filter(col("row_num") == 1) \
        .drop("row_num")

    # Instancia a tabela destino para fazer o MERGE (Evita duplicar se reprocessado)
    df_deduped.write.format("delta") \
        .mode("append") \
        .option("mergeSchema", "true") \
        .saveAsTable(cfg_click["target_table"])

# Disparando a escrita da stream do Clickstream
query_click = df_silver_click_parsed.writeStream \
    .format("delta") \
    .foreachBatch(upsert_clickstream_silver) \
    .option("checkpointLocation", cfg_click["checkpoint"]) \
    .trigger(availableNow=True) \
    .start()        

query_click.awaitTermination()
print(f" ✅ Clickstream limpo e unificado em: {cfg_click['target_table']}")    

# =====
# --- PIPELINE sales
# =====

df_bronze_sales = spark.readStream.table(cfg_sales["source_table"])

# Padronização de tipos de dados básicos
df_sales_typed = df_bronze_sales.select(
    col("transaction_id").cast("string"),
    col("user_id").cast("string"),
    col("product_id").cast("string"),
    col("amount").cast("double"),
    col("payment_method").cast("string"),
    col("transaction_timestamp").cast("timestamp"),
    current_timestamp().alias("silver_processing_timestamp")
)

# Aplica a Arquitetura de Quarentena Avançada via foreachBatch
def route_sales_by_quality(df_micro_batch, batch_id):
    if df_micro_batch.count() == 0:
        return
        
    # Elimina IDs nulos e Valores negativas ou zeros
    condicao_valida = col("transaction_id").isNotNull() & (col("amount") > 0)
    
    # Divide o dataframe em valores e inválidos para gravar em tabelas separadas    
    df_trusted    = df_micro_batch.filter(condicao_valida)
    df_quarantine = df_micro_batch.filter(~condicao_valida)
    
    # Grava os dados limpos na tabela Silver Trusted
    if df_trusted.count() > 0:
        df_trusted.write.format("delta").mode("append").saveAsTable(cfg_sales["target_table"])
        
    # Desvia o lixo/anomalias para a tabela de Quarentena sem derrubar o pipeline principal
    if df_quarantine.count() > 0:
        print(f"⚠️ {df_quarantine.count()} registros ilegais identificados e desviados para a Quarentena!")
        df_quarantine.write.format("delta").mode("append").saveAsTable(cfg_sales["quarantine_table"])

# Disparando a escrita da stream de Sales
query_sales = df_sales_typed.writeStream \
    .format("delta") \
    .foreachBatch(route_sales_by_quality) \
    .option("checkpointLocation", cfg_sales["checkpoint"]) \
    .trigger(availableNow=True) \
    .start()

query_sales.awaitTermination()
print(f" ✅ Sales processado com sucesso!")
print(f"   -> Trusted: {cfg_sales['target_table']}")
print(f"   -> Quarentena: {cfg_sales['quarantine_table']}")

print("\n✨ Camada Silver concluída com sucesso! Os dados agora são totalmente confiáveis.")